# Probability of precipitation type (%) — ENS point meteogram

Python/earthkit port of the Metview macro *"PROBABILITY OF PRECIPITATION TYPE (%)"* (E. Gascón, 12/01/2017).

For one location and one ENS run it shows, at every output step, the percentage of members forecasting each
precipitation type (`ptype`, param 260015) split into three precipitation-rate classes (`tprate`, param 260048, mm/h):
*low* (type threshold – 0.2), *mid* (0.2 – 1) and *high* (> 1).

**Differences from the macro (for speed):**
- MARS data are retrieved with `earthkit.data` on a small box around the location (native grid, no interpolation) instead of the whole globe.
- The request is split by member chunks and run in parallel — MARS cropping of native-grid fields is CPU-bound and single-threaded per request.
- Downloaded GRIB is cached in `OUT_DIR`; any cached file with a superset of the requested steps is reused.
- Point extraction and probabilities are vectorised with NumPy; the chart is an `earthkit.plots` time-series figure (ECMWF style and logo) with stacked bars.
- Since IFS 50r1 (12 May 2026) ENFO has no control forecast, so the member count is taken from the data instead of being fixed at 51.

In [ ]:
import shutil
from concurrent.futures import ThreadPoolExecutor
from datetime import date, timedelta
from pathlib import Path

import earthkit.data as ekd
import earthkit.plots as ekp
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator, PercentFormatter

## User parameters

In [ ]:
OUT_DIR = Path("/perm/moeg/Products_plotting")   # GRIB cache + plots

FDATE = (date.today() - timedelta(days=1)).strftime("%Y%m%d")   # YYYYMMDD
TIME = "00"            # 00 / 06 / 12 / 18
EXPVER = "1"
DAYS = 7               # 1 … 15
MIN_STEP = 1           # 1 / 3 / 6 / 12 (h) – ENS is hourly to T+90, 3-hourly to T+144, 6-hourly after

LAT, LON = 50.73, 7.1  # New York 40.72,-74 · Reading 51.38,-0.97 · Bologna 44.49,11.34
CITY, COUNTRY = "Bonn", "Germany"

# [N, W, S, E]. None -> 1° box around the location. A larger box (e.g. Europe) costs about
# the same MARS time and lets several cities be plotted from one cached download.
AREA = None

# Lower precipitation-rate threshold (mm/h) of the "low" class; wet snow uses the snow threshold.
THRESHOLDS = {
    "rain": 0.12,
    "snow": 0.04,
    "freezrain": 0.05,
    "freezdrizzle": 0.03,
    "sleet": 0.1,
    "icepellets": 0.05,
}

N_PARALLEL = 8                       # concurrent MARS requests
INCLUDE_CF = FDATE < "20260512"      # IFS 50r1 removed the control forecast from ENFO

## Constants and helpers

In [ ]:
PARAM_PTYPE, PARAM_PRATE, PARAM_Z, PARAM_LSM = 260015, 260048, 129, 172
# older cycles have no total rate (260048): use the sum of convective/large-scale rain and snowfall rates
PRATE_COMPONENTS = [228218, 228219, 228220, 228221]   # crr, lsrr, csfr, lssfr
RATE_BINS = (0.2, 1.0)   # mm/h: low | mid | high
CLASS_NAMES = ("low", "mid", "high")

# Legend order (left -> right); the bars are stacked in reverse order (freezing rain at the bottom).
# Colours are (low, mid, high), taken from the macro (Magics named colours approximated in RGB).
PTYPES = {
    "rain":         dict(code=1,  label="Rain",             thr="rain",
                         colours=[(0.79, 0.94, 0.75), (0.23, 0.89, 0.08), (0.10, 0.64, 0.03)]),
    "sleet":        dict(code=7,  label="Sleet",            thr="sleet",
                         colours=[(0.479, 0.917, 0.866), (0.25, 0.88, 0.82), (0.081, 0.444, 0.402)]),
    "wetsnow":      dict(code=6,  label="Wet snow",         thr="snow",
                         colours=[(0.53, 0.81, 0.98), (0.268, 0.268, 0.979), (0.0, 0.0, 0.5)]),
    "snow":         dict(code=5,  label="Snow",             thr="snow",
                         colours=[(0.829, 0.757, 0.945), (0.58, 0.43, 0.82), (0.336, 0.111, 0.697)]),
    "icepellets":   dict(code=8,  label="Ice pellets",      thr="icepellets",
                         colours=[(0.923, 0.727, 0.544), (1.0, 0.65, 0.0), (0.74, 0.39, 0.02)]),
    "freezdrizzle": dict(code=12, label="Freezing drizzle", thr="freezdrizzle",
                         colours=[(0.988, 0.737, 0.882), (0.988, 0.435, 0.757), (0.722, 0.271, 0.533)]),
    "freezrain":    dict(code=3,  label="Freezing rain",    thr="freezrain",
                         colours=[(0.96, 0.51, 0.54), (1.0, 0.01, 0.29), (0.75, 0.02, 0.30)]),
}


def compute_steps(min_step: int, days: int) -> list[int]:
    fstep = days * 24
    segments = [(0, 90, min_step), (93, 144, max(min_step, 3)), (150, fstep, max(min_step, 6))]
    return [s for start, end, inc in segments for s in range(start, min(end, fstep) + 1, inc)]


def nearest_index(fieldlist, lat: float, lon: float) -> int:
    """Index of the nearest grid point by great-circle distance (as Metview's nearest_gridpoint)."""
    geo = fieldlist[0].geography
    lats, lons = np.radians(geo.latitudes()), np.radians(geo.longitudes())
    lat0, lon0 = np.radians(lat), np.radians(lon)
    hav = np.sin((lats - lat0) / 2) ** 2 + np.cos(lat0) * np.cos(lats) * np.sin((lons - lon0) / 2) ** 2
    return int(np.argmin(hav))


def grid_point(fieldlist, idx: int) -> tuple[float, float]:
    geo = fieldlist[0].geography
    return float(geo.latitudes()[idx]), float(geo.longitudes()[idx])


def fmt_latlon(lat: float, lon: float) -> str:
    lon = (lon + 180) % 360 - 180
    return f"{abs(lat):.2f}°{'N' if lat >= 0 else 'S'} {abs(lon):.2f}°{'E' if lon >= 0 else 'W'}"

## Retrieve (or reuse cached) ENS data

In [ ]:
STEPS = compute_steps(MIN_STEP, DAYS)
if AREA is None:
    AREA = [round(LAT + 0.5, 2), round(LON - 0.5, 2), round(LAT - 0.5, 2), round(LON + 0.5, 2)]

OUT_DIR.mkdir(parents=True, exist_ok=True)
area_tag = "_".join(f"{v:g}" for v in AREA)
run_tag = f"{FDATE}{TIME}_e{EXPVER}"
ENS_FILE = OUT_DIR / f"ptype_prate_{run_tag}_{DAYS}d_{MIN_STEP}s_{area_tag}.grib"
OROG_FILE = OUT_DIR / f"orog_lsm_{run_tag}_{area_tag}.grib"

# Reuse any cached file of the same run/area whose steps cover the requested ones
if not ENS_FILE.exists():
    for cand in sorted(OUT_DIR.glob(f"ptype_prate_{run_tag}_*_{area_tag}.grib")):
        cached_steps = set(ekd.from_source("file", str(cand)).to_fieldlist().get("metadata.step"))
        if cached_steps.issuperset(STEPS):
            ENS_FILE = cand
            break

BASE_REQUEST = {"class": "od", "stream": "enfo", "expver": EXPVER, "date": FDATE,
                "time": TIME, "levtype": "sfc", "area": AREA}


def mars_to_file(request: dict, target: Path) -> Path:
    ekd.from_source("mars", {**BASE_REQUEST, **request}).to_target("file", str(target))
    return target


def available_prate_params() -> list[int]:
    probe = {**BASE_REQUEST, "type": "pf", "number": 1, "step": STEPS[0], "param": PARAM_PRATE}
    try:
        ekd.from_source("mars", probe).to_fieldlist()
        return [PARAM_PRATE]
    except Exception:   # MARS client fails with "Expected 1, got 0" when the parameter is not archived
        return PRATE_COMPONENTS


tasks = []
ens_parts = []
if not ENS_FILE.exists():
    params = [PARAM_PTYPE, *available_prate_params()]
    print("Parameters:", params)
    ens_requests = [{"type": "pf", "number": chunk.tolist(), "param": params, "step": STEPS}
                    for chunk in np.array_split(np.arange(1, 51), N_PARALLEL)]
    if INCLUDE_CF:
        ens_requests.append({"type": "cf", "param": params, "step": STEPS})
    ens_parts = [ENS_FILE.with_suffix(f".part{i}") for i in range(len(ens_requests))]
    tasks += list(zip(ens_requests, ens_parts))
if not OROG_FILE.exists():
    tasks.append(({"type": "pf", "number": 1, "param": [PARAM_Z, PARAM_LSM], "step": 0}, OROG_FILE))

if tasks:
    print(f"Retrieving {len(tasks)} MARS request(s) in parallel …")
    with ThreadPoolExecutor(N_PARALLEL) as pool:
        list(pool.map(lambda t: mars_to_file(*t), tasks))
    if ens_parts:
        # write to a temporary name first so a failed run never leaves a partial cache file
        tmp = ENS_FILE.with_suffix(".tmp")
        with open(tmp, "wb") as out:
            for part in ens_parts:
                with open(part, "rb") as f:
                    shutil.copyfileobj(f, out)
                part.unlink()
        tmp.rename(ENS_FILE)

print("ENS file :", ENS_FILE)
print("Orog file:", OROG_FILE)

## Extract the nearest grid point and compute probabilities

In [ ]:
fl = ekd.from_source("file", str(ENS_FILE)).to_fieldlist()
param, number, step = np.array(fl.get(["metadata.paramId", "metadata.number", "metadata.step"]), dtype=int).T

idx = nearest_index(fl, LAT, LON)
gp_lat, gp_lon = grid_point(fl, idx)

# decode only the fields of the requested steps, keeping just the value at the grid point
sel = np.flatnonzero(np.isin(step, STEPS))
param, number, step = param[sel], number[sel], step[sel]
point = np.array([fl[int(i)].to_numpy(flatten=True)[idx] for i in sel])

steps = np.unique(step)
members = np.unique(number)
m_idx, s_idx = np.searchsorted(members, number), np.searchsorted(steps, step)

ptype = np.full((members.size, steps.size), np.nan)
is_pt = param == PARAM_PTYPE
ptype[m_idx[is_pt], s_idx[is_pt]] = point[is_pt]

# total rate = sum over the rate parameter(s) in the file (260048, or its 4 components)
prate = np.zeros((members.size, steps.size))
n_rate = np.zeros((members.size, steps.size), dtype=int)
np.add.at(prate, (m_idx[~is_pt], s_idx[~is_pt]), point[~is_pt] * 3600.0)   # kg m-2 s-1 -> mm/h
np.add.at(n_rate, (m_idx[~is_pt], s_idx[~is_pt]), 1)

missing = np.isnan(ptype) | (n_rate != np.unique(param[~is_pt]).size)
if missing.any():
    raise RuntimeError(f"Incomplete data: {missing.sum()} member/step values missing in {ENS_FILE}")

ofl = ekd.from_source("file", str(OROG_FILE)).to_fieldlist()
oidx = nearest_index(ofl, LAT, LON)
z, lsm = (ofl.sel(**{"metadata.paramId": p})[0].to_numpy(flatten=True)[oidx] for p in (PARAM_Z, PARAM_LSM))
orog_m = z * lsm / 9.80665

n_mem = members.size
code = np.rint(ptype)
prob = {}   # name -> (3, n_steps) array of % for (low, mid, high)
for name, p in PTYPES.items():
    is_type = code == p["code"]
    low = is_type & (prate >= THRESHOLDS[p["thr"]]) & (prate < RATE_BINS[0])
    mid = is_type & (prate >= RATE_BINS[0]) & (prate < RATE_BINS[1])
    high = is_type & (prate >= RATE_BINS[1])
    prob[name] = 100.0 * np.stack([low, mid, high]).sum(axis=1) / n_mem

base_time = pd.Timestamp(f"{FDATE} {TIME}:00")
valid = base_time + pd.to_timedelta(steps, unit="h")

df = pd.DataFrame({(name, cls): prob[name][k] for name in PTYPES for k, cls in enumerate(CLASS_NAMES)},
                  index=pd.Index(valid, name="valid_time"))
print(f"{n_mem} members, {steps.size} steps, grid point {fmt_latlon(gp_lat, gp_lon)}, {orog_m:.0f} m")
df.loc[:, (df > 0).any()].round(1)

## Plot

In [ ]:
x = np.array(valid.to_pydatetime())
spacing_h = np.diff(steps, append=steps[-1] + (steps[-1] - steps[-2] if steps.size > 1 else MIN_STEP))
width = 0.85 * spacing_h / 24.0   # bar width in days, follows the output frequency

figure = ekp.Figure(rows=1, columns=1, figsize=(11.69, 8.27))   # A4 landscape
ts = figure.add_timeseries()

bottom = np.zeros(steps.size)
for name in reversed(PTYPES):
    for k in (2, 1, 0):   # high, mid, low
        h = prob[name][k]
        nz = h > 0
        if nz.any():
            ts.bar(None, x=x[nz], y=h[nz], width=width[nz], bottom=bottom[nz], align="center",
                   color=PTYPES[name]["colours"][k], edgecolor="black", linewidth=0.4)
        bottom += h

# earthkit-plots has no multi-column grouped legend or date-tick control, so these use the underlying axes
ax = ts.ax
handles, labels = [], []
for name, p in PTYPES.items():
    thr = THRESHOLDS[p["thr"]]
    handles += [Patch(visible=False)] + [Patch(facecolor=p["colours"][k], edgecolor="black", linewidth=0.5)
                                         for k in (2, 1, 0)]
    labels += [f"{p['label']} (mm/h)", ">1", "0.2-1", f"{thr:g}-0.2"]
leg = ax.legend(handles, labels, ncol=len(PTYPES), loc="lower center", bbox_to_anchor=(0.5, 1.0),
                frameon=False, fontsize=11, handlelength=1.8, handleheight=1.2, columnspacing=0.8,
                handletextpad=0.4)
for text in leg.get_texts()[::4]:
    text.set_fontweight("bold")

ax.set_ylim(0, 100)
ax.set_xlim(x[0] - pd.Timedelta(days=width[0]), x[-1] + pd.Timedelta(days=width[-1]))
ax.yaxis.set_major_locator(MultipleLocator(10))
ax.yaxis.set_major_formatter(PercentFormatter())
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=range(0, 24, 6)))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d\n%b %Y"))
ax.grid(True, which="major", color="grey", linestyle="--", linewidth=0.6)
ax.set_axisbelow(True)

figure.title(
    f"Base time: {base_time:%Y%m%d %H} UTC to +{steps[-1]} h\n"
    f"Probability of precipitation type (%) depending on precipitation rates (mm/h) — ENS forecast at "
    f"{fmt_latlon(LAT, LON)} {CITY} ({COUNTRY}), {orog_m:.0f} m",
    fontsize=11, y=1.08,
)
figure.add_logo("ecmwf")

out_png = OUT_DIR / f"Prob_prec_type_{CITY.replace(' ', '_')}_{run_tag}_{DAYS}d_{MIN_STEP}s.png"
figure.save(out_png, dpi=150)
print("Saved", out_png)
figure.show()